---
- **Cifar-100**
    - 해상도 : 32x32x3
    - class : 100개
    - train data : 50000개 (500images x 100 class)
    - test data : 10000개  (100images x 100 class)

--- 
고전 방식

- 모델설계 : torchvision
- 학습루프 : nn.Module
- 데이터처리 : torchvision

이렇게 3개로 먼저 해봤음

https://www.youtube.com/watch?v=AA621UofTUA

---

### **data 가져오기 및 특수토큰, 파라미터 정의**

In [17]:
from dataclasses import dataclass, field
from typing import List
import torch

@dataclass
class TransformerConfig:
    data_path: str = "./Multi30k/data/task1/raw" # 또는 /tok
    src_lang: str = "de" # 독일어 동화 같은 느낌임.
    tgt_lang: str = "en" # 영어
    
    # 특수 토큰 인덱스
    PAD_IDX: int = 0  # <pad> : 문장 길이 맞추기
    SOS_IDX: int = 1  # <SOS> : Start
    EOS_IDX: int = 2  # <EOS> : End
    UNK_IDX: int = 3  # <Unk> : Unknown : 빈도가 낮은 단어가 변환된다?
    # 위 4가지 특수 토큰의 문자열 리스트
    special_symbols: List[str] = field(default_factory=lambda: ["<pad>", "<sos>", "<eos>", "<unk>"])
    
    # 모델 하이퍼파라미터 (나중에 모델 정의 시 사용)
    batch_size: int = 512
    d_model: int = 512  # d_model
    n_head: int = 8     # multi-head attention을 수행하기위한 head 개수 (d_k = d_model/head -> 각 head의 차원)
    num_encoder_layers: int = 6 # N
    num_decoder_layers: int = 6 # N
    dim_feedforward: int = 2048 # attention 뒤에 붙는 MLP (보통 d_model의 4배 정도)
    dropout: float = 0.1
    device: torch.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(device)

cuda


---

### **DataLoader Customize**

In [18]:
import os
from torch.utils.data import Dataset
import spacy
from collections import Counter

class Multi30KDataset(Dataset):
    def __init__(self, config: TransformerConfig, split='train', src_vocab=None, tgt_vocab=None):
        self.config = config
        self.split = split
        
        # 1. 토크나이저 설정 (spacy 모델 : 문장을 단어 단위로 쪼개는 도구)
        #self.src_tokenizer = get_tokenizer('spacy', language='de_core_news_sm')
        #self.tgt_tokenizer = get_tokenizer('spacy', language='en_core_web_sm')
        self.de_nlp = spacy.load('de_core_news_sm')
        self.en_nlp = spacy.load('en_core_web_sm')
        # 토크나이저 함수 정의
        self.src_tokenizer = lambda text: [tok.text for tok in self.de_nlp.tokenizer(text)]
        self.tgt_tokenizer = lambda text: [tok.text for tok in self.en_nlp.tokenizer(text)]

        # 2. 로컬 파일 읽기
        # train.de, train.en 열어서 가져오기
        src_file = os.path.join(config.data_path, f"{split}.{config.src_lang}")
        tgt_file = os.path.join(config.data_path, f"{split}.{config.tgt_lang}")
        
        with open(src_file, 'r', encoding='utf-8') as f:
            self.src_data = [line.strip() for line in f] # "한 줄씩 담아서 가져옴"
        with open(tgt_file, 'r', encoding='utf-8') as f:
            self.tgt_data = [line.strip() for line in f]
            
        assert len(self.src_data) == len(self.tgt_data), "Source와 Target 데이터 개수가 다릅니다!"

        # 3. 단어 사전(Vocab) 처리 (test, infernce 차이)
        # train이면 직접 vocab을 만들고, val/test면 train에서 학습하며 만든 만든 vocab을 받아옵니다.
        if split == 'train':
            self.src_vocab = self._build_vocab(self.src_data, self.src_tokenizer)
            self.tgt_vocab = self._build_vocab(self.tgt_data, self.tgt_tokenizer)
        else:
            self.src_vocab = src_vocab
            self.tgt_vocab = tgt_vocab

    # 단어 사전 (vocab) 만들기용
    def _build_vocab(self, data_list, tokenizer): # data와 torkenizer 도구를 받음.
        counter = Counter()
        for text in data_list: # data에 있는 text를 lower case로 변환하고 단어별로 중복되는 개수를 센다
            counter.update(tokenizer(text.lower()))
        
        # 특수 토큰 먼저 할당 (pad, sos, eos, unk 순서로 vocab에 담음 0, 1, 2, 3)
        vocab = {token: i for i, token in enumerate(self.config.special_symbols)}
        # 빈도수 2 이상인 단어만 추가 (min_freq 조절 가능)
        for word, freq in counter.items(): # 즉, 빈도가 2번 이상인 단어만 vocab에 담음!
            if freq >= 2 and word not in vocab:
                vocab[word] = len(vocab) # 순서대로 vocab에 담기 시작 -> 특수토큰 다음인 index 4부터 시작.
        return vocab

    def __len__(self):
        # enumerate와 DataLoader가 멈출 시점을 결정합니다.
        return len(self.src_data)

    def __getitem__(self, idx):
        # 1. idx에 해당하는 en, de 문장 가져오기
        src_text = self.src_data[idx]
        tgt_text = self.tgt_data[idx]
        
        # 2. 토큰화 및 수치화
        src_tokens = self.src_tokenizer(src_text.lower())
        tgt_tokens = self.tgt_tokenizer(tgt_text.lower())
        
        # 3. tokenizer를 통해 쪼개진 단어들을 vocab에 등록된 번호로 바꿈.
        #     빈도수가 1 이하인 사전에 등록되지 않은 단어는 3번(Unk) 특수토큰으로 바꿔줌.
        src_indices = [self.src_vocab.get(token, self.config.UNK_IDX) for token in src_tokens]
        tgt_indices = [self.tgt_vocab.get(token, self.config.UNK_IDX) for token in tgt_tokens]
        
        # 4. SOS, EOS 추가 및 텐서 변환
        src_indices = [self.config.SOS_IDX] + src_indices + [self.config.EOS_IDX]
        tgt_indices = [self.config.SOS_IDX] + tgt_indices + [self.config.EOS_IDX]
        
        return torch.tensor(src_indices), torch.tensor(tgt_indices)

In [19]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

# batch안에 채울때 길이를 맞추기 위해 사용
def collate_fn(batch):
    src_batch, tgt_batch = [], []
    for src_sample, tgt_sample in batch:
        src_batch.append(src_sample)
        tgt_batch.append(tgt_sample)

    # padding_value는 Config에 정의한 PAD_IDX(0)를 사용
    src_batch = pad_sequence(src_batch, padding_value=0) # pad_sequence : 가장 긴 문장을 기준으로 짧은 문장들의 빈칸을 padding_value=0 로 채워줌.
    tgt_batch = pad_sequence(tgt_batch, padding_value=0)
    
    return src_batch, tgt_batch

# Dataloader 작동 방식.
# 1. Dataset의 getitem을 배치크기만큼 호출해서 sample을 리스트로 모아옴
# 2. 이 리스트를 collate_fn에 줌
# 3. collate_fn은 이 샘플들을 하나로 묶어서 큰 텐서로 만듬
config = TransformerConfig()
train_dataset = Multi30KDataset(config, split='train')
train_loader = DataLoader(train_dataset, batch_size=config.batch_size, 
                          shuffle=True, collate_fn=collate_fn)

print(f"독일어 사전 크기: {len(train_dataset.src_vocab)}")
print(f"영어 사전 크기: {len(train_dataset.tgt_vocab)}")

독일어 사전 크기: 7853
영어 사전 크기: 5893


---

### **positional Encoding + Embedding**

In [20]:
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # 1. 위치 정보를 담을 행렬 초기화 (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        
        # 2. 위치 인덱스 생성 (0, 1, 2, ..., max_len-1)
        # shape: (max_len, 1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # 3. 주기 함수에 들어갈 계수(div_term) 계산
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        # 4. 짝수 인덱스(2i)에는 sin, 홀수 인덱스(2i+1)에는 cos 적용
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # 5. 차원 맞추기: (max_len, 1, d_model) 
        pe = pe.unsqueeze(0).transpose(0, 1)
        
        # 6. 학습되지 않는 파라미터이므로 register_buffer로 등록
        self.register_buffer('pe', pe)

    def forward(self, x):

        # x의 길이만큼만 위치 정보를 가져와서 더해줌
        x = x + self.pe[:x.size(0), :]
        
        # 논문에서 언급된 대로 마지막에 Dropout 적용
        return self.dropout(x)

# Embedding + PE
class TransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        # Embedding: 단어를 d_model 차원의 벡터로 변환
        self.token_emb = nn.Embedding(vocab_size, d_model)
        
        # Positional Encoding: 아까 정의했던 PositionalEncoding 클래스 사용
        self.pos_emb = PositionalEncoding(d_model, dropout, max_len)
        
        self.d_model = d_model

    def forward(self, x):
        # Embedding 후 sqrt(d_model)을 곱해줌으로써 scaling
        out = self.token_emb(x) * math.sqrt(self.d_model)
        # positional 정보 더하기
        out = self.pos_emb(out)
        
        return out # 최종 shape: (Seq_Len, Batch_Size, d_model)

---

### **Multi-head attention, FFN**

In [27]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.1):
        super().__init__()
        assert d_model % n_head == 0 # d_model이 head로 나눠지는지 확인
        
        self.d_model = d_model
        self.n_head = n_head
        self.d_k = d_model // n_head # 512 / 8 = 64 (head별 dimension)
        
        # 1. 가중치 행렬들 (W_q, W_k, W_v)
        # 512x512로 만들고 나중에 64x64로 쪼갬.
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model) # 최종 출력용 Wo
        
        self.dropout = nn.Dropout(dropout)
        self.scale = torch.sqrt(torch.FloatTensor([self.d_k])) # 루트(d_k)

    def forward(self, q, k, v, mask=None):
        batch_size = q.shape[1] # (Seq, Batch, Dim) 기준이므로 index 1에서 가져옴.
        
        # Step 1: Linear 연산 수행
        Q = self.w_q(q)  # q(Lx512) @ W_q(512X512)
        K = self.w_k(k)
        V = self.w_v(v)
        
        # Step 2: 헤드 쪼개기
        # (Seq, Batch, 512) -> (Seq, Batch, 8, 64) -> (Batch, 8, Seq, 64)
        # transpose(0, 1) 등을 써서 Batch가 맨 앞으로 오게 정렬합니다 (연산 편의성)
        # 즉, Lx 512 -> Lx64 8channel
        Q = Q.view(-1, batch_size, self.n_head, self.d_k).permute(1, 2, 0, 3) 
        K = K.view(-1, batch_size, self.n_head, self.d_k).permute(1, 2, 0, 3)
        V = V.view(-1, batch_size, self.n_head, self.d_k).permute(1, 2, 0, 3)
        
        # Step 3: Scaled Dot-Product Attention 계산 (Q @ Kt -> LxL)
        # (Q @ Kt) / root d_k
        # Q,K,V = (Batch, Head, Seq_len_q, d_k) -> (B, 8, L_q, 64)
        # K.permute(0, 1, 3, 2) -> (B, 8, 64, L_k)
        # matmul은 앞의 (B, 8)은 건드리지 않고, 마지막 두 차원끼리만 행렬 곱을 수행 (L_q x 64 @ 64 x L_k -> L_q x L_k)
        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale.to(q.device)
        
        # Decoder의 Masked Multi-head attention
        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e10) # 마스크 씌우기 (-무한대)

        # energy shape : [Batch_size, n_head, Seq_len_q(L_q), Seq_len_k(L_k)]    
        attention = self.dropout(torch.softmax(energy, dim=-1)) # softmax 통과 후 dropout
        
        # 결과물: (Batch, 8, Seq, 64)
        x = torch.matmul(attention, V) # attention과 V 행렬곱으로 최종 attention값 구함
        
        # Step 4: 다시 합치기 (Concat)
        # (Batch, 8, Seq, 64) -> (Seq, Batch, 512)
        # Lx 64 8 channel -> L x 512
        x = x.permute(2, 0, 1, 3).contiguous()
        x = x.view(-1, batch_size, self.d_model)
        
        return self.w_o(x), attention # 마지막 w_o 통과하여 return

In [28]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff) # 512 -> 2048
        self.w_2 = nn.Linear(d_ff, d_model) # 2048 -> 512
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (Seq_len, Batch, 512)
        x = self.dropout(torch.relu(self.w_1(x)))
        x = self.w_2(x)
        return x        # w_2(dropout(relu(w_1*x)))

---

### **Encoder, Decoder**

In [29]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout):
        super().__init__()

        # multi-head attention 부분
        self.self_attn = MultiHeadAttention(d_model, n_head, dropout) # Multi head attention
        self.self_attn_layer_norm = nn.LayerNorm(d_model)   # MHA Layer Norm
        
        # FFN 부분
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout) # FFN
        self.ffn_layer_norm = nn.LayerNorm(d_model) # FFN layernorm
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_mask):
        # --- Step 1: Self-Attention Block ---
        # Residual Connection: x + Sublayer(x)
        _src, _ = self.self_attn(src, src, src, src_mask)
        src = self.self_attn_layer_norm(src + self.dropout(_src))
        
        # --- Step 2: Feed-Forward Block ---
        # Residual Connection: x + Sublayer(x)
        _src = self.ffn(src)
        src = self.ffn_layer_norm(src + self.dropout(_src))
        
        return src

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_head, d_ff, dropout, max_len):
        super().__init__()
        self.embedding = TransformerEmbedding(vocab_size, d_model, dropout, max_len)
        
        # EncoderLayer를 n_layers번 반복해서 쌓습니다.
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_head, d_ff, dropout) 
            for _ in range(n_layers)
        ])

    def forward(self, src, src_mask):
        x = self.embedding(src)
        for layer in self.layers:
            x = layer(x, src_mask)
        return x

In [30]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff, dropout):
        super().__init__()
        
        # 1. Self-Attention (미래 단어를 못 보게 마스킹함-> Decoder QKV)
        self.self_attn = MultiHeadAttention(d_model, n_head, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        
        # 2. Encoder-Decoder Attention (인코더 KV를 받아옴)
        self.cross_attn = MultiHeadAttention(d_model, n_head, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        
        # 3. FFN
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm3 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, trg, enc_src, trg_mask, src_mask):
        # Step 1: Masked Self-Attention
        # trg: 디코더 입력 (지금까지 생성한 단어들)
        _trg, _ = self.self_attn(trg, trg, trg, trg_mask)
        trg = self.norm1(trg + self.dropout(_trg))
        
        # Step 2: Cross-Attention (핵심!)
        # Query는 디코더(trg)에서, Key와 Value는 인코더(enc_src)에서 가져옵니다.
        # "내(디코더)가 지금 이 단어를 쓰려는데, 인코더 정보 중 어디를 봐야 할까?"
        _trg, attention = self.cross_attn(trg, enc_src, enc_src, src_mask)
        trg = self.norm2(trg + self.dropout(_trg))
        
        # Step 3: Feed Forward
        _trg = self.ffn(trg)
        trg = self.norm3(trg + self.dropout(_trg))
        
        return trg, attention

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_head, d_ff, dropout, max_len):
        super().__init__()
        # 1. 타겟 언어(예: 영어)를 위한 임베딩 + 위치 정보
        self.embedding = TransformerEmbedding(vocab_size, d_model, dropout, max_len)
        
        # 2. DecoderLayer를 n_layers번 쌓음
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_head, d_ff, dropout)
            for _ in range(n_layers)
        ])

    def forward(self, trg, enc_src, trg_mask, src_mask):
        # trg: [Seq_len_trg, Batch_size]
        # enc_src: [Seq_len_src, Batch_size, d_model] (인코더의 최종 출력)
        
        x = self.embedding(trg)
        
        # 층층이 통과하며 Cross-Attention 수행
        for layer in self.layers:
            # attention 결과값도 필요하면 저장해서 시각화할 수 있습니다.
            x, attention = layer(x, enc_src, trg_mask, src_mask)
            
        return x, attention

---

### **Encoder + Decoder 조립해서 Transformer로 만들자** 

In [31]:
class Transformer(nn.Module):
    def __init__(self, encoder, decoder, src_pad_idx, trg_pad_idx, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.device = device
        
        # 최종 출력: d_model 차원을 단어 사전 크기(tgt_vocab_size)로 투영
        self.fc_out = nn.Linear(encoder.embedding.token_emb.embedding_dim, 
                                decoder.embedding.token_emb.num_embeddings)

    def make_src_mask(self, src):
        # src: [Seq_len, Batch] -> [Batch, 1, 1, Seq_len]
        # 패딩(0)인 부분은 False, 실제 단어는 True인 마스크 생성
        src_mask = (src != self.src_pad_idx).transpose(0, 1).unsqueeze(1).unsqueeze(2)
        return src_mask

    def make_trg_mask(self, trg):
        # 1. 패딩 마스크 (인코더와 동일)
        trg_pad_mask = (trg != self.trg_pad_idx).transpose(0, 1).unsqueeze(1).unsqueeze(2)
        
        # 2. Look-ahead 마스크 (미래 단어 가리기)
        trg_len = trg.shape[0]
        # 상삼각형 행렬을 만들어 미래 단어 위치를 0으로 가림
        trg_sub_mask = torch.tril(torch.ones((trg_len, trg_len), device=self.device)).bool()
        
        # 두 마스크를 합침 (둘 다 만족해야 함)
        trg_mask = trg_pad_mask & trg_sub_mask
        return trg_mask

    def forward(self, src, trg):
        # 마스크 생성
        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)
        
        # 인코더 통과
        enc_src = self.encoder(src, src_mask)
        
        # 디코더 통과 (인코더 결과 enc_src를 함께 넣음)
        out, attention = self.decoder(trg, enc_src, trg_mask, src_mask)
        
        # 최종 단어 예측
        return self.fc_out(out), attention

---

**모델테스트**

In [32]:
import torch

# 1. 하이퍼파라미터 설정 (테스트용)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

INPUT_DIM = 7853  # 독어 사전 크기
OUTPUT_DIM = 5893 # 영어 사전 크기
D_MODEL = 512
N_LAYERS = 3      # 테스트니까 가볍게 3층만
N_HEADS = 8
D_FF = 2048
DROPOUT = 0.1
MAX_LEN = 100

# 패딩 인덱스 정의 (보통 1번이나 0번)
SRC_PAD_IDX = 1
TRG_PAD_IDX = 1

# 2. 모델 인스턴스 생성
enc = Encoder(INPUT_DIM, D_MODEL, N_LAYERS, N_HEADS, D_FF, DROPOUT, MAX_LEN)
dec = Decoder(OUTPUT_DIM, D_MODEL, N_LAYERS, N_HEADS, D_FF, DROPOUT, MAX_LEN)

model = Transformer(enc, dec, SRC_PAD_IDX, TRG_PAD_IDX, device).to(device)

# 3. 가짜 데이터 생성 (Batch_size=32, Seq_len=20)
# 각 숫자는 단어 번호를 의미함
src = torch.randint(0, INPUT_DIM, (20, 32)).to(device)
trg = torch.randint(0, OUTPUT_DIM, (15, 32)).to(device) # trg_input 역할

# 4. 모델 실행
output, attention = model(src, trg)

# 5. 결과 확인
print(f"입력 src 모양: {src.shape}")       # [20, 32]
print(f"입력 trg 모양: {trg.shape}")       # [15, 32]
print("-" * 30)
print(f"출력 output 모양: {output.shape}")  # [15, 32, 5893]
print(f"어텐션 맵 모양: {attention.shape}") # [32, 8, 15, 20]

입력 src 모양: torch.Size([20, 32])
입력 trg 모양: torch.Size([15, 32])
------------------------------
출력 output 모양: torch.Size([15, 32, 5893])
어텐션 맵 모양: torch.Size([32, 8, 15, 20])
